# Business Dimension Table - Gold Layer

## Overview
Builds a business dimension to track historical changes in business attributes over time. Preserves full history of changes to holder names, addresses, areas, and licensing requirements.

## Purpose
* Create business dimension for dimensional model with historical tracking
* Track changes to business attributes over time (SCD Type 2)
* Join with street dimension to denormalize street_id
* Support point-in-time queries ("What was the business area on this date?")
* Enable audit trail of business attribute changes

## Key Features
**SCD Type 2** - maintains full history of attribute changes  
**Change detection** - automatically detects when attributes change  
**Version management** - closes old versions, inserts new versions  
**Point-in-time queries** - supports historical analysis  
**Street dimension join** - denormalizes street_id for performance


## Output Schema
**Table**: `workspace.tel_aviv_municipality.dim_business`

| Column | Type | Description |
|--------|------|-------------|
| `business_id` | LONG | Natural key (business identifier) |
| `primary_holder_name` | STRING | License holder name (slowly changing) |
| `street_id` | INT | Foreign key to dim_street (slowly changing) |
| `house_number` | INT | Street address number (slowly changing) |
| `house_area_sqm` | DOUBLE | Property area in sqm (slowly changing) |
| `type` | STRING | Business type/category |
| `is_licensing_required` | BOOLEAN | Licensing requirement (slowly changing) |
| `foundation_date` | DATE | Business foundation date (slowly changing) |
| `effective_from` | TIMESTAMP | When this version became effective |
| `effective_to` | TIMESTAMP | When this version expired (NULL = current) |
| `is_current` | BOOLEAN | TRUE for current version, FALSE for historical |

**Grain**: One row per business per version (multiple versions per business)

**Staged Updates Pattern**:
1. Join new data with current table to find changes
2. Create staged_updates (rows that changed) with merge_key=null
3. Union with original data (merge_key=business_id)
4. Merge: match on merge_key to close old, insert new versions

## Integration
**Upstream**: 
* `stg.business` (silver layer - business data)
* `dim_street` (gold layer - street dimension for street_id lookup)

**Downstream**: 
* `fact_daily_business_compensation` (joins on business_id with temporal conditions)
* Analytics queries requiring historical business state

## Notes
* First run creates all records as current (is_current=true)
* Subsequent runs detect changes and version accordingly
* Multiple changes over time create multiple versions per business
* Always join on `business_id` with temporal filter for point-in-time accuracy
* `effective_from` sourced from `_ingestion_at` in staging layer

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
table_name = "workspace.tel_aviv_municipality.dim_business"

# Prepare Data from Stg
new_business_data = spark.table("workspace.tel_aviv_municipality_stg.business").alias("biz") \
    .join(spark.table("workspace.tel_aviv_municipality.dim_street").alias("st"), "street_name", "left") \
    .select(
        F.col("biz.business_id"),
        F.col("biz.primary_holder_name"),
        F.col("st.street_id"),
        F.col("biz.house_number"),
        F.col("biz.house_area_sqm"),
        F.col("biz.type"),
        F.col("biz.is_licensing_required"), 
        F.col("biz.foundation_date"),       
        F.col("biz._ingestion_at").alias("effective_from")
    )

# Initialize Table if it doesn't exist
if not spark.catalog.tableExists(table_name):
    new_business_data \
        .withColumn("effective_to", F.lit(None).cast("timestamp")) \
        .withColumn("is_current", F.lit(True)) \
        .write.format("delta").saveAsTable(table_name)
else:
    # Merge Logic
    target_table = DeltaTable.forName(spark, table_name)
    
    # Define the change condition: If any of these differ, we create a new version
    change_condition = """
        t.is_current = true AND (
            s.street_id <> t.street_id OR 
            s.primary_holder_name <> t.primary_holder_name OR
            s.is_licensing_required <> t.is_licensing_required OR
            s.foundation_date <> t.foundation_date OR
            s.house_area_sqm <> t.house_area_sqm
        )
    """

    # Identify updates (rows that changed) to be inserted as new versions
    staged_updates = new_business_data.alias("s") \
        .join(spark.table(table_name).alias("t"), "business_id") \
        .where(change_condition) \
        .select("s.*") \
        .withColumn("merge_key", F.lit(None)) 

    # Combine with original data (merge_key = ID allows closing old records)
    finish_updates = new_business_data.withColumn("merge_key", F.col("business_id")) \
        .unionByName(staged_updates, allowMissingColumns=True)

    (target_table.alias("t")
     .merge(finish_updates.alias("s"), "t.business_id = s.merge_key")
     .whenMatchedUpdate(
         condition=change_condition,
         set={
             "is_current": "false",
             "effective_to": "s.effective_from"
         }
     )
     .whenNotMatchedInsert(values={
         "business_id": "s.business_id",
         "primary_holder_name": "s.primary_holder_name",
         "street_id": "s.street_id",
         "house_number": "s.house_number",
         "house_area_sqm": "s.house_area_sqm",
         "type": "s.type",
         "is_licensing_required": "s.is_licensing_required", 
         "foundation_date": "s.foundation_date",            
         "effective_from": "s.effective_from",
         "effective_to": "null",
         "is_current": "true"
     })
     .execute())